In [ ]:
import random
import time
from collections import deque
import threading
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- CSS Styling ---
custom_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono&display=swap');

:root {
    --primary: #004ac6;
    --primary-dark: #003ea8;
    --text: #141b2b;
    --muted: #54647a;
    --line: #c3c6d7;
    --soft: #eef1ff;
    --soft-2: #f7f8ff;
}

.app-container {
    background-color: #f9f9ff;
    font-family: 'Inter', sans-serif;
    border-radius: 16px;
    overflow: hidden;
}
.modern-card {
    background-color: #ffffff;
    border: 1px solid var(--line);
    border-radius: 14px;
    padding: 20px;
    box-shadow: 0 1px 3px rgba(20, 27, 43, 0.05);
    box-sizing: border-box;
}
.card-header {
    font-size: 18px;
    font-weight: 700;
    color: var(--text);
    border-bottom: 1px solid var(--line);
    padding-bottom: 10px;
    margin-bottom: 15px;
    display: flex;
    justify-content: space-between;
    align-items: center;
}
.card-subtitle {
    font-size: 12px;
    color: var(--muted);
    line-height: 1.5;
    margin-bottom: 12px;
}
.stat-box {
    background-color: #ffffff !important;
    border: 1px solid var(--line) !important;
    border-radius: 14px !important;
    padding: 15px !important;
    box-sizing: border-box;
    box-shadow: 0 1px 3px rgba(20, 27, 43, 0.04);
}
.puzzle-input input[type="number"] {
    font-size: 24px !important;
    font-weight: 700 !important;
    text-align: center !important;
    color: #004ac6 !important;
    background-color: #e1e8fd !important;
    border: 1px solid rgba(0,74,198,0.2) !important;
    border-radius: 8px !important;
    height: 100% !important;
    box-sizing: border-box;
    box-shadow: none !important;
}
.puzzle-input input[type="number"]::-webkit-outer-spin-button,
.puzzle-input input[type="number"]::-webkit-inner-spin-button {
    -webkit-appearance: none;
    margin: 0;
}
.puzzle-input input[type="number"] {
    -moz-appearance: textfield;
}
.btn-primary {
    background-color: var(--primary) !important;
    color: white !important;
    border-radius: 999px !important;
    font-weight: 700 !important;
    border: 1px solid var(--primary) !important;
    width: 95% !important;
    box-sizing: border-box !important;
}
.btn-primary:hover { background-color: var(--primary-dark) !important; }
.btn-action {
    background-color: #e1e8fd !important;
    color: #38485d !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
    border: 1px solid rgba(0,74,198,0.2) !important;
    font-size: 12px !important;
}
.btn-action:hover { opacity: 0.9 !important; }
.log-output {
    background-color: #f1f3ff !important;
    font-family: 'JetBrains Mono', monospace !important;
    border: none !important;
}

/* Visual Simulation giống ảnh mẫu */
.anim-board {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 14px;
    max-width: 330px;
    margin: 0 auto;
    background-color: #eef1ff;
    padding: 24px;
    border-radius: 20px;
    position: relative;
    box-shadow: inset 0 2px 10px rgba(20, 27, 43, 0.03);
}
.anim-tile {
    aspect-ratio: 1;
    background-color: #ffffff;
    border: 1px solid #c3c6d7;
    border-radius: 14px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 36px;
    font-weight: 800;
    color: #004ac6;
    box-shadow: 0 3px 10px rgba(20, 27, 43, 0.11);
    transition: all 0.3s ease;
}
.anim-tile-empty {
    aspect-ratio: 1;
    background-color: rgba(225, 232, 253, 0.55);
    border: 2px dashed #b7bfd8;
    border-radius: 14px;
    box-sizing: border-box;
}
.badge {
    background:#e1e8fd;
    padding: 4px 10px;
    border-radius: 6px;
    font-size: 12px;
    font-weight: 700;
    color:#004ac6;
}
.badge-ok { background:#d1f4e0; color:#0d6e35; }
.badge-error { background:#ffe4e4; color:#b42318; }
</style>
"""

# --- UI Components ---
display(HTML(custom_css))

# Header Thanh tiêu đề chính
header_html = widgets.HTML(value="""
<div style="display: flex; justify-content: space-between; align-items: center; padding: 16px 30px; background-color: #ffffff; border-bottom: 1px solid #c3c6d7; font-family: 'Inter', sans-serif;">
    <span style="font-size: 20px; font-weight: 800; color: #141b2b;">8-Puzzle Solver Simulator</span>
    <div style="color: #004ac6; font-weight: 800; border-bottom: 2px solid #004ac6; padding-bottom: 4px; font-size: 14px; letter-spacing: 0.4px;">MIN-CONFLICTS</div>
</div>
""")

# 1. Khu vực Initial State (Bên trái)
input_boxes = [widgets.BoundedIntText(value=v, min=0, max=8, layout=widgets.Layout(width='auto', height='60px')) 
               for v in [1, 2, 3, 4, 0, 5, 7, 8, 6]]
for box in input_boxes: box.add_class('puzzle-input')

input_grid = widgets.GridBox(input_boxes, layout=widgets.Layout(grid_template_columns="repeat(3, 1fr)", gap="10px", margin="0 0 20px 0"))

btn_random = widgets.Button(description="Random", layout=widgets.Layout(flex='1'))
btn_random.add_class('btn-action')
btn_random.style.button_color = '#505f76'
btn_random.style.text_color = 'white'

btn_reset = widgets.Button(description="Reset", layout=widgets.Layout(flex='1'))
btn_reset.add_class('btn-action')
btn_load = widgets.Button(description="Load Example", layout=widgets.Layout(flex='1'))
btn_load.add_class('btn-action')

action_btns = widgets.HBox([btn_random, btn_reset, btn_load], layout=widgets.Layout(gap='10px'))

initial_state_card = widgets.VBox([
    widgets.HTML('<div class="card-header"><span>1. Initial State</span></div>'),
    input_grid,
    action_btns
], layout=widgets.Layout(margin='0 0 20px 0'))
initial_state_card.add_class('modern-card')

# 2. Khu vực Min-Conflicts Configuration
btn_min_conflicts = widgets.Button(description="Run Min-Conflicts", layout=widgets.Layout(height='45px', flex='1'))
btn_min_conflicts.add_class('btn-primary')

limit_slider = widgets.IntSlider(
    value=40,
    min=1,
    max=100,
    step=1,
    description="Limit:",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="95%")
)

max_steps_slider = widgets.IntSlider(
    value=1000,
    min=100,
    max=5000,
    step=100,
    description="Max steps:",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="95%")
)

config_card = widgets.VBox([
    widgets.HTML('<div class="card-header"><span>2. Min-Conflicts Configuration</span></div>'),
    widgets.HTML('<div class="card-subtitle">Conflict được hiểu là trạng thái trung gian chưa nối hợp lệ với trạng thái trước/sau. Thuật toán chọn biến đang conflict và đổi sang giá trị ít conflict nhất.</div>'),
    widgets.VBox([limit_slider, max_steps_slider, btn_min_conflicts], layout=widgets.Layout(width='100%', align_items='center', gap='10px'))
], layout=widgets.Layout(margin='0 0 20px 0')) 
config_card.add_class('modern-card')

# 3. Khu vực Visual Simulation 
step_label = widgets.HTML('<span class="badge">Ready</span>')
sim_header = widgets.HBox([
    widgets.HTML('<span style="font-size: 18px; font-weight: 600; color: #141b2b;">3. Visual Simulation</span>'),
    step_label
], layout=widgets.Layout(justify_content='space-between', border_bottom='1px solid #c3c6d7', padding='0 0 15px 0', margin='0 0 20px 0', width='100%'))

anim_html = widgets.HTML(value="")
sim_card = widgets.VBox([sim_header, anim_html], layout=widgets.Layout(flex='1'))
sim_card.add_class('modern-card')

# Gom nhóm cột trái 
left_col = widgets.VBox([initial_state_card, sim_card], layout=widgets.Layout(flex='1.6', min_width='50%'))

# 4. Khu vực Thống kê (Stats Grid)
stat_steps = widgets.HTML()
stat_nodes = widgets.HTML()
stat_time = widgets.HTML()
stat_depth = widgets.HTML()

def update_stat(html_widget, value):
    html_widget.value = f'<div style="font-size: 28px; font-weight: 700; color: #141b2b;">{value}</div>'

update_stat(stat_steps, "-1")
update_stat(stat_nodes, "-1")
update_stat(stat_time, "-1")
update_stat(stat_depth, "-1")

def make_stat_box(title, html_widget):
    box = widgets.VBox([
        widgets.HTML(f'<span style="font-size: 11px; font-weight: 600; color: #54647a; text-transform: uppercase;">{title}</span>'),
        html_widget
    ], layout=widgets.Layout(width='100%'))
    box.add_class('stat-box')
    return box

stat_grid = widgets.GridBox([
    make_stat_box("Steps", stat_steps),
    make_stat_box("Nodes", stat_nodes),
    make_stat_box("Time", stat_time),
    make_stat_box("Max Depth", stat_depth)
], layout=widgets.Layout(grid_template_columns="1fr 1fr", gap="15px", margin="0 0 20px 0"))

# Khu vực Log kết quả chạy thuật toán
log_content = widgets.HTML(value='')
out_text = widgets.VBox([log_content], layout=widgets.Layout(flex='1', overflow='auto', padding='15px', max_height='350px'))
out_text.add_class('log-output')

log_card = widgets.VBox([
    widgets.HTML('<div style="display:flex; align-items:center; justify-content:space-between; border-bottom: 1px solid #c3c6d7; padding: 12px 20px; background-color: #e1e8fd; border-radius: 12px 12px 0 0;"><span style="font-size: 13px; font-weight: 700; color: #141b2b; text-transform: uppercase; letter-spacing: 0.5px;">Execution Log</span><div style="display:flex; gap: 5px;"><span style="color:#54647a; font-size:16px;">📋</span><span style="color:#54647a; font-size:16px;">⬇️</span></div></div>'),
    out_text
], layout=widgets.Layout(background_color='#f1f3ff', border='1px solid #c3c6d7', border_radius='12px', flex='1'))

# Xếp nhóm cột phải
right_col = widgets.VBox([
    stat_grid, 
    config_card, 
    log_card
], layout=widgets.Layout(flex='1', min_width='330px'))

# Tổng hợp ứng dụng vào container chính
main_app = widgets.VBox([
    header_html,
    widgets.HBox([left_col, right_col], layout=widgets.Layout(padding='20px', gap='20px'))
])
main_app.add_class('app-container')

# --- Logic Core ---
def get_successors(mt):
    pos = mt.index(0)
    r, c = pos // 3, pos % 3
    successors = []
    
    def swap(mt, i, j):
        new_mt = list(mt)
        new_mt[i], new_mt[j] = new_mt[j], new_mt[i]
        return new_mt
        
    if c > 0: successors.append(("Trái", swap(mt, pos, pos - 1)))
    if c < 2: successors.append(("Phải", swap(mt, pos, pos + 1)))
    if r > 0: successors.append(("Lên", swap(mt, pos, pos - 3)))
    if r < 2: successors.append(("Xuống", swap(mt, pos, pos + 3)))
    return successors

def is_successor(s1, s2):
    try:
        pos1 = s1.index(0)
        pos2 = s2.index(0)
        r1, c1 = pos1 // 3, pos1 % 3
        r2, c2 = pos2 // 3, pos2 % 3
        dist = abs(r1 - r2) + abs(c1 - c2)
        if dist != 1:
            return False
        temp = list(s1)
        temp[pos1], temp[pos2] = temp[pos2], temp[pos1]
        return temp == s2
    except:
        return False

def get_action(s1, s2):
    for action, child in get_successors(s1):
        if child == s2:
            return action
    return None

def compute_conflicts(assignment, i):
    v = assignment[i]
    v_prev = assignment[i-1]
    v_next = assignment[i+1]
    conf = 0
    if not is_successor(v_prev, v):
        conf += 1
    if not is_successor(v, v_next):
        conf += 1
    return conf

def min_conflicts_for_k(start_state, goal_state, k, max_steps=1000):
    if k == 1:
        if is_successor(start_state, goal_state):
            act = get_action(start_state, goal_state)
            return [(act, goal_state)], 1
        else:
            return None, 1
            
    assignment = [list(start_state)]
    for i in range(1, k):
        prev = assignment[i-1]
        succs = [s for _, s in get_successors(prev)]
        assignment.append(random.choice(succs))
    assignment.append(list(goal_state))
    
    nodes_generated = k
    
    for step in range(max_steps):
        all_ok = True
        for i in range(k):
            if not is_successor(assignment[i], assignment[i+1]):
                all_ok = False
                break
        if all_ok:
            path = []
            for i in range(k):
                act = get_action(assignment[i], assignment[i+1])
                path.append((act, assignment[i+1]))
            return path, nodes_generated
            
        conflicted_vars = []
        for i in range(1, k):
            if compute_conflicts(assignment, i) > 0:
                conflicted_vars.append(i)
                
        var_idx = random.choice(conflicted_vars)
        v_prev = assignment[var_idx - 1]
        v_next = assignment[var_idx + 1]
        
        candidates = []
        seen = set()
        for _, s in get_successors(v_prev):
            t = tuple(s)
            if t not in seen:
                candidates.append(s)
                seen.add(t)
        for _, s in get_successors(v_next):
            t = tuple(s)
            if t not in seen:
                candidates.append(s)
                seen.add(t)
                
        current_val = assignment[var_idx]
        if tuple(current_val) not in seen:
            candidates.append(current_val)
            
        best_val = []
        min_conf = 3
        
        for cand in candidates:
            orig = assignment[var_idx]
            assignment[var_idx] = cand
            conf = compute_conflicts(assignment, var_idx)
            assignment[var_idx] = orig
            
            if conf < min_conf:
                min_conf = conf
                best_val = [cand]
            elif conf == min_conf:
                best_val.append(cand)
        
        chosen_val = random.choice(best_val)
        if chosen_val != assignment[var_idx]:
            nodes_generated += 1
        assignment[var_idx] = chosen_val
        
    return None, nodes_generated

def min_conflicts_search(start_state, goal_state, limit, max_steps=1000):
    if start_state == goal_state:
        return [], 1
        
    for k in range(1, limit + 1):
        path, nodes = min_conflicts_for_k(start_state, goal_state, k, max_steps=max_steps)
        if path is not None:
            return path, nodes
    return None, 0

def in_mt(mt):
    res = ""
    for i in range(3):
        row = ""
        for j in range(3):
            val = mt[i*3 + j]
            if val == 0: row += " [ ] "
            else: row += f"  {val}  "
        res += row + "\n"
    res += "-" * 20 + "\n"
    return res

def print_log_state(title, state, action=None):
    state_str = ""
    for i in range(0, 9, 3):
        row = state[i:i+3]
        state_str += "  " + "    ".join([str(x) if x != 0 else "[ ]" for x in row]) + "\n"
    
    if action:
        action_html = f'<p style="margin-bottom: 8px; font-weight: 600; color: #141b2b;">Bước {title}: Di chuyển ô trống sang <span style="color: #004ac6;">{action}</span></p>'
    else:
        action_html = f'<p style="margin-bottom: 8px; font-weight: 600; color: #141b2b;">{title}</p>'
        
    log_html = f'''
    <div style="margin-bottom: 15px;">
        {action_html}
        <div style="background-color: #ffffff; padding: 12px; border-radius: 8px; border: 1px solid rgba(195, 198, 215, 0.5); display: inline-block;">
            <pre style="margin: 0; font-family: 'JetBrains Mono', monospace; font-size: 13px; line-height: 1.4; color: #141b2b;">{state_str}</pre>
        </div>
    </div>
    <div style="border-top: 1px solid rgba(195, 198, 215, 0.5); margin-bottom: 15px; width: 100%;"></div>
    '''
    log_content.value += log_html

def render_board(state):
    html_content = '<div class="anim-board">'
    for val in state:
        if val == 0: html_content += '<div class="anim-tile-empty"></div>'
        else: html_content += f'<div class="anim-tile">{val}</div>'
    html_content += '</div>'
    anim_html.value = html_content

def animate_path(start_state, path):
    final_state = path[-1][1] if path else start_state
    current = [0] * 9

    render_board(current)
    step_label.value = f'<span class="badge">Step 0/8</span>'
    time.sleep(0.8)

    reveal_step = 0
    for idx, val in enumerate(final_state):
        if val == 0:
            continue
        reveal_step += 1
        current[idx] = val
        render_board(current)
        step_label.value = f'<span class="badge">Step {reveal_step}/8</span>'
        time.sleep(0.55)

    step_label.value = '<span class="badge badge-ok">Finished</span>'


def solve_in_background(start_state, goal_state, limit, max_steps):
    # Chờ một chút để người dùng thấy bảng đã bị xóa hết và stats đang là -1
    time.sleep(0.6)

    log_title = "MIN-CONFLICTS SEARCH"
    start_time = time.time()
    path, nodes_generated = min_conflicts_search(start_state, goal_state, limit, max_steps=max_steps)
    elapsed_ms = int((time.time() - start_time) * 1000)

    log_content.value += f'<div style="color: #004ac6; font-weight: bold; margin-bottom: 10px; font-family: Inter;">ĐANG GIẢI BẰNG: {log_title}</div>'
    log_content.value += f'<div style="color: #54647a; margin-bottom: 10px; font-family: Inter; font-size: 13px;">Limit = <b>{limit}</b>, Max steps mỗi k = <b>{max_steps}</b></div>'

    if path is not None:
        update_stat(stat_steps, str(len(path)))
        update_stat(stat_nodes, f"{nodes_generated:,}")
        update_stat(stat_time, f"{elapsed_ms}ms")
        update_stat(stat_depth, str(len(path)))

        print_log_state("Trạng thái ban đầu:", start_state)
        if path:
            print_log_state("Trạng thái kết quả:", path[-1][1])
        log_content.value += '<div style="color:#004ac6; font-weight:600; margin-bottom:10px; font-family:Inter;">Visual Simulation: xóa bảng rồi hiện từng ô số một, không di chuyển ô trống.</div>'

        animate_path(start_state, path)
    else:
        log_content.value += '<div style="color: red; font-weight: bold; font-family: Inter;">Không tìm thấy giải pháp!</div>'
        update_stat(stat_steps, "-1")
        update_stat(stat_nodes, f"{nodes_generated:,}")
        update_stat(stat_time, f"{elapsed_ms}ms")
        update_stat(stat_depth, "-1")
        step_label.value = '<span class="badge badge-error">Failed</span>'


def solve_and_animate():
    log_content.value = ''

    # Giống AC-3: bấm chạy là reset toàn bộ bảng thống kê về -1
    update_stat(stat_steps, "-1")
    update_stat(stat_nodes, "-1")
    update_stat(stat_time, "-1")
    update_stat(stat_depth, "-1")

    # Giống AC-3: bấm chạy là xóa hết số trong Visual Simulation ngay lập tức
    render_board([0] * 9)
    step_label.value = '<span class="badge">Running</span>'

    start_state = [box.value for box in input_boxes]
    goal_state = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    limit = int(limit_slider.value)
    max_steps = int(max_steps_slider.value)

    thread = threading.Thread(
        target=solve_in_background,
        args=(start_state, goal_state, limit, max_steps),
        daemon=True
    )
    thread.start()

btn_min_conflicts.on_click(lambda b: solve_and_animate())

def randomize_board(b):
    nums = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    random.shuffle(nums)
    for i, box in enumerate(input_boxes): box.value = nums[i]
    render_board(nums)
    log_content.value = ''
    step_label.value = '<span class="badge">Ready</span>'
btn_random.on_click(randomize_board)

def reset_board(b):
    nums = [0]*9
    for i, box in enumerate(input_boxes): box.value = nums[i]
    render_board(nums)
    for stat in [stat_steps, stat_nodes, stat_time, stat_depth]:
        update_stat(stat, "-1")
    log_content.value = ''
    step_label.value = '<span class="badge">Ready</span>'
btn_reset.on_click(reset_board)

def load_example(b):
    nums = [1, 2, 3, 4, 0, 5, 7, 8, 6]
    for i, box in enumerate(input_boxes): box.value = nums[i]
    render_board(nums)
    log_content.value = ''
    step_label.value = '<span class="badge">Ready</span>'
btn_load.on_click(load_example)

# Khởi chạy ban đầu
render_board([1, 2, 3, 4, 0, 5, 7, 8, 6])
display(main_app)